# Connect a Python function

See how a Python function becomes a Method, receives selected data and returns a result.

**Write a function → register a Method → select inputs → query data → inspect the result**

A product revenue ranking shows the basic flow. OLS regression then shows how to connect several variables. Saved results appear below the code using prepared example data.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# examples/ is outside the installed decision_layer package.
repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "examples/methods").is_dir()
)
sys.path.insert(0, str(repo_root))

from decision_layer.methods import Column, DataInput, Scope, python_method
from importlib import reload
from examples.methods.authoring import customer_analysis

# Read updated example code when rerunning this setup cell.
practice_session = reload(customer_analysis).practice_session

## 1. Rank products by revenue

A small ranking function shows how the function and its data connect.

### Calculation function

Rank the `value` column and return it alongside `group`. The function does not need to know about Cube or Decision Layer.

In [2]:
def rank_values(data, descending=True):
    """Rank observed values, retaining ties."""
    return data[["group", "value"]].assign(
        rank=data["value"].rank(method="min", ascending=not descending).astype(int)
    )


frame = pd.DataFrame({"group": ["A", "B", "C"], "value": [30, 20, 20]})
standalone = rank_values(frame)
standalone

,group,value,rank
0,A,30,1
1,B,20,2
2,C,20,2


### Register the Method

`python_method(...)` gives the function a name and declares its inputs.

- `name`, `version`: the Method's identity and version
- `DataInput`: the data passed to the function
- `Column`: a required column. Use `measure` for a metric and `dimension` for a dimension.

Registration declares a `value` metric and a `group` dimension. The actual revenue metric and product dimension are selected at execution time.

`practice_session()` creates a local session with prepared example data. `session.register(...)` adds the Method to that session.

In [3]:
rank_method = python_method(
    rank_values,
    name="my_team.rank",
    version="1.0.0",
    inputs={
        "data": DataInput(
            columns={
                "value": Column(source="measure"),
                "group": Column(source="dimension"),
            },
        ),
    },
)

session = await practice_session()
session.register(rank_method)
print("Registered Methods:", [method.name for method in session.list_methods()])

등록된 Method: ['my_team.rank']


### Select inputs and execute

Connect the actual data for a product revenue ranking.

| Setting | Selection in this example |
|---|---|
| `bindings` | Revenue for `value`, product for `group` |
| `params` | `descending=True`: highest revenue first |
| `scope` | Date range and date dimension |

Decision Layer queries the selected data, builds a DataFrame with `value` and `group` columns, and passes it to `rank_values`.

An AI request follows the same flow. AI selects a registered Method and semantic inputs, asks about ambiguous metrics or dates, and explains the result after Decision Layer queries and calculates it.

In [4]:
trial = await session.run(
    "my_team.rank",
    bindings={
        "value": "cube://shop/sales/revenue",
        "group": "cube://shop/sales/product",
    },
    params={"descending": True},
    scope=Scope(
        date_range=("2026-09-01", "2026-09-30"),
        time_dimension="cube://shop/sales/date",
    ),
)

trial.result.primary.data

[{'group': 'A', 'value': 30.0, 'rank': 1},
 {'group': 'B', 'value': 20.0, 'rank': 2},
 {'group': 'C', 'value': 20.0, 'rank': 2}]

## 2. OLS with several variables

The ranking introduced how a function connects to data. Reusing the same method matters more when code handles several variables and analytical conditions.

- A data scientist registers reviewed code once.
- Users and AI connect the required data to reuse that analysis.
- Decision Layer records server inputs and results in a Run.

Next, OLS examines the relationship between customer ad exposures and purchase amounts.

### Calculation function

Examine ad exposures and purchases while also accounting for age and prior purchases.

- `data`: the DataFrame to analyze
- `outcome`: the purchase amount column
- `treatment`: the ad exposure column
- `confounders`: the age and prior purchase amount columns

Column names are arguments, so the function can be reused with different data. This example uses numeric synthetic data. **The result describes association; it does not establish causation.**

In [5]:
def ols(data, outcome, treatment, confounders):
    """Use OLS to examine a variable and outcome while accounting for other variables."""
    import statsmodels.api as sm

    design = sm.add_constant(data[[treatment, *confounders]], has_constant="add")
    fit = sm.OLS(data[outcome], design).fit()
    return {
        "value": float(fit.params[treatment]),
        "n_observations": int(fit.nobs),
        "r_squared": float(fit.rsquared),
        "coefficients": {key: float(value) for key, value in fit.params.items()},
        "causal_claim": "not_established",
        "uncertainty": "not_reported",
    }

### Check analysis conditions

Conditions to check before calculation can live in a separate function. Here, `check_ols` checks missing values and the regression design.

Checks are optional. Keep a short check in `method.py`, or move longer checks to `checks.py`. **The filename alone does not trigger execution.**


In [ ]:
def check_ols(data, outcome, treatment, confounders):
    """Check the data and variables needed for this OLS calculation."""
    import numpy as np
    import pandas as pd
    import statsmodels.api as sm

    columns = [outcome, treatment, *confounders]
    if len(set(columns)) != len(columns):
        raise ValueError("The same column was selected more than once.")

    values = data[columns]
    if not all(pd.api.types.is_numeric_dtype(values[col]) for col in columns):
        raise ValueError("Select numeric columns only.")
    if not np.isfinite(values.to_numpy(dtype=float)).all():
        raise ValueError("Handle missing or infinite values first.")
    if values[outcome].nunique() < 2:
        raise ValueError("The outcome has no variation.")

    design = sm.add_constant(values[[treatment, *confounders]], has_constant="add")
    if len(data) <= design.shape[1]:
        raise ValueError("Add more observations or select fewer variables.")
    if np.linalg.matrix_rank(design) < design.shape[1]:
        raise ValueError(
            "Some predictors are exact linear combinations of others. Review the selected variables."
        )

### Call it directly in Python

For a direct Python call, run the check and calculation in order.


In [6]:
from examples.methods.authoring.customer_analysis import customer_sample

customers = customer_sample()

check_ols(
    customers, "purchase_amount", "ad_exposures", ["age", "prior_purchase_amount"]
)

direct_ols = ols(
    customers,
    outcome="purchase_amount",
    treatment="ad_exposures",
    confounders=["age", "prior_purchase_amount"],
)

direct_ols

{'value': 50.00000000000206,
 'n_observations': 200,
 'r_squared': 1.0,
 'coefficients': {'const': 999.999999999618,
  'ad_exposures': 50.00000000000206,
  'age': 9.999999999999813,
  'prior_purchase_amount': 0.20000000000000567},
 'causal_claim': 'not_established',
 'uncertainty': 'not_reported'}

### Register the Method

`entity_key` identifies each customer row. `multiple=True` lets `confounders` accept several variables.

Connect the check with `checks=[check_ols]`. Decision Layer then follows this sequence:

**Query data → check the input contract → run `check_ols` → call `ols` if the check passes**

The check receives the same DataFrame and column names as the calculation. If it raises `ValueError` or returns `False`, execution stops with a refused result and the check's reason. `ols` is not called.

This function returns a dict with a numeric `value`, so its result becomes an estimate automatically. Use `output="table"` if you want a table instead.


In [7]:
my_ols = python_method(
    ols,
    name="my_team.ols",
    version="1.0.0",
    inputs={
        "data": DataInput(
            columns={
                "entity_id": Column(source="dimension", role="entity_key"),
                "outcome": Column(source="measure", dtype="number"),
                "treatment": Column(
                    source=["dimension", "measure"],
                    dtype="number",
                ),
                "confounders": Column(
                    source=["dimension", "measure"],
                    dtype="number",
                    multiple=True,
                ),
            },
            minimum_rows=10,
        )
    },
    checks=[check_ols],
    requires_packages={"statsmodels": ">=0.14,<0.16"},
)

session.register(my_ols)
print("Registered Methods:", [method.name for method in session.list_methods()])

등록된 Method: ['my_team.rank', 'my_team.ols']


### Select inputs and execute

Connect the customer ID, purchases and ad exposures, then select age and prior purchases as additional variables.

In [8]:
# Examine ad exposures and purchases, accounting for age and prior purchases.
ols_trial = await session.run(
    "my_team.ols",
    bindings={
        "entity_id": "cube://shop/customers/id",
        "outcome": "cube://shop/customers/purchase_amount",
        "treatment": "cube://shop/customers/ad_exposures",
        "confounders": [
            "cube://shop/customers/age",
            "cube://shop/customers/prior_purchase_amount",
        ],
    },
    scope=Scope(
        date_range=("2026-02-01", "2026-02-28"),
        time_dimension="cube://shop/customers/date",
    ),
)

ols_trial.result.primary.data

{'value': 50.00000000000206,
 'n_observations': 200,
 'r_squared': 1.0,
 'coefficients': {'const': 999.999999999618,
  'treatment': 50.00000000000206,
  'confounders_0': 9.999999999999813,
  'confounders_1': 0.20000000000000567},
 'causal_claim': 'not_established',
 'uncertainty': 'not_reported'}

### How selected inputs reach the function

**1. Check the inputs.**

Decision Layer checks that the selected metrics and dimensions exist in Cube and satisfy the Method's input requirements.

| Input | Selected data |
|---|---|
| `entity_id` | Customer ID |
| `outcome` | Purchase amount |
| `treatment` | Ad exposures |
| `confounders` | Age, prior purchase amount |

**2. Query customer-level data from Cube.**

The selected metrics, dimensions and dates become a Cube query. Cube returns customer-level data using the aggregations and joins defined in its model.

<details>
<summary>View the Cube request</summary>

This JSON is sent to `/cubejs-api/v1/load`.

```json
{
  "query": {
    "dimensions": ["customers.id", "customers.age"],
    "measures": [
      "customers.purchase_amount",
      "customers.ad_exposures",
      "customers.prior_purchase_amount"
    ],
    "timeDimensions": [
      {
        "dimension": "customers.date",
        "dateRange": ["2026-02-01", "2026-02-28"]
      }
    ],
    "order": [["customers.id", "asc"]],
    "limit": 50000,
    "offset": 0
  }
}
```

</details>

**3. Rename the columns and pass them to `ols`.**

Purchase amount becomes `outcome`, and ad exposures become `treatment`. Age and prior purchase amount become `confounders_0` and `confounders_1`, in the order selected.

| entity_id | outcome | treatment | confounders_0 | confounders_1 |
|---|---:|---:|---:|---:|
| C001 | 8340 | 10 | 40 | 32200 |
| C002 | 10110 | 6 | 69 | 40600 |

Decision Layer calls the function with this table and its column names.

```python
ols(
    data=queried_dataframe,
    outcome="outcome",
    treatment="treatment",
    confounders=["confounders_0", "confounders_1"],
)
```


## 3. How AI selects data through MCP

AI reads registered Methods with `list_methods` and checks the chosen Method with `describe_method`. It then calls **`get_semantic_catalog()` to read all available definitions** and selects inputs by their titles, descriptions and data types. It does not need a search term or prior knowledge of internal Cube names.

The catalog comes from Cube `/meta`. In this example it contains both customer and sales definitions. AI chooses customer purchase amount, ad exposures, age and prior purchases for the OLS inputs; it asks when the intended meaning is ambiguous. Actual customer rows are requested only when the analysis executes.

The function below defines client calls without making a network request. Calling it requires a connected MCP client, the Method installed in the server project and the `shop` Cube connection. Registering a Method in this notebook only affects the local session.

In [ ]:
async def inspect_with_mcp(client):
    await client.call_tool("list_methods", {})
    method = await client.call_tool("describe_method", {"name": "my_team.ols"})
    catalog = await client.call_tool("get_semantic_catalog", {})
    return method.structured_content, catalog.structured_content

After selecting inputs, AI creates a Run with `start_analysis` and passes the chosen refs to `run_step`. The server queries Cube and calls the registered function, using the same bindings and calculation path as the local example above. AI reads the recorded evidence with `get_run` and saves its explanation with `complete_run`.

See [MCP connection](../../guides/mcp.md) for setup.

## Use it in a project

The notebook defines functions in cells. A project can split them into files:

```text
methods/ols/
├── method.py        # Calculation and python_method registration
├── checks.py        # Optional check functions
└── test_method.py   # Tests for the calculation and checks
```

After moving the check, import it in `method.py` with `from .checks import check_ols` and pass it through `checks=[check_ols]`, as above. The server discovers `method.py`.

`test_method.py` runs during development with `decision-layer test`. It is separate from checks that run with each analysis.

Continue with [Register in a project](../../guides/method-projects.md) for file layout and server startup.
